# Помесячные выходы договоров в NPL: когорты 2023–2026

**Вход:** любые месячные Excel-портфели (`.xlsx`, `.xlsm`) — в файлах могут быть десятки других колонок. При загрузке выбираются **только 18 столбцов из согласованного списка**; остальные игнорируются (не загружаются в DataFrame).

**Разрешённые 18 полей:** `УНП`, `Клиент`, `Сегмент`, `Номер договора`, `Тип операции`, `Код валюты`, `ГР %`, `%рез`, `проср задолж, тыс BYN`, `Задолженность, тыс BYN`, `Резерв факт`, `Резерв рассчит`, `Курс`, `Max-Day OD`, `Max-Day %%`, `Дата начала`, `Дата окончания`, `NPL(договор)`.

Для расчёта обязательно наличие **7 ключевых колонок**: `УНП`, `Клиент`, `Сегмент`, `Номер договора`, `Задолженность, тыс BYN`, `Дата начала`, `NPL(договор)`. Если остальные 11 разрешённых полей отсутствуют — это допустимо, они не нужны для расчёта. Пропуски отмечаются на листе «Контроль_файлы». Регистр, пробелы, переносы строк и знаки пунктуации в заголовках нормализуются; поддерживаются несколько вариантов названий.

**NPL строго числовой:** `0` (в том числе `0.0`) = нет NPL; **любое число, отличное от нуля**, включая отрицательное = NPL. Пустые, текстовые или некорректные значения приводят к явной ошибке с названием файла: их нельзя молча объявлять NPL или не-NPL.

**Выход в NPL:** договор был **в предыдущем месячном портфеле с NPL=0**, а в текущем месячном портфеле получил **NPL≠0**. Для этого перехода берём **сумму задолженности на конец месяца** из `Задолженность, тыс BYN` **по всем строкам договора** (валютам, типам операций). Договор идентифицируется парой **УНП + Номер договора**. Строки одного договора агрегируются до сравнения NPL.

Далее результаты группируются по **месяцу перехода**, **году заключения договора** (год `Дата начала`) и двум сегментам: **Средний+** и **Малый+Микро**.

- Договоры 2023 года: выходы в 2023–2026 годах; договоры 2024 года: в 2024–2026; договоры 2025 года: в 2025–2026; договоры 2026 года: в 2026.
- Срезы — конец каждого месяца. Для расчёта января 2023 требуется декабрь 2022; аналогично на границах остальных лет.
- Если предыдущего **соседнего** среза нет, соответствующий месяц **пропускается**, а не отражается как ноль.
- Если договор **отсутствовал** в предыдущем срезе и появился сразу с NPL≠0, это **не доказанный переход 0→1**: такой случай не включается, а отмечается в контроле.
- Повторный вход одного договора в NPL после выхода из NPL учитывается как **новое месячное событие**.
- Денежная единица отчёта — **тыс. BYN**; умножение на 1000 не выполняется.

## 1. Настройки

Поместите файлы в одну папку. Примеры: `2022-12.xlsx`, `2023-01.xlsx` ... `2026-12.xlsx` или `портфель_31.01.2023.xlsx`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ОБЯЗАТЕЛЬНО: измените путь
DATA_DIR = Path(r"/путь/к/папке/портфели")
OUTPUT_FILE = DATA_DIR / "Выходы_NPL_по_когортам_2023_2026.xlsx"

SHEET_NAME = 0                  # Номер или имя листа, например "Портфель"
HEADER_ROW = 0                  # 0 = первая строка Excel
CONTRACT_COL = "Номер договора"  # Используем только это поле из согласованных 18
MIN_COHORT_YEAR = 2023
MAX_COHORT_YEAR = 2026

# Если имя файла не содержит однозначной даты, впишите соответствие здесь.
# Пример: {"portfolio_A.xlsx": "2023-01", "portfolio_B.xlsx": "2023-02"}
FILE_MONTH_OVERRIDE = {}

# Важно: редактируйте только если реальные наименования сегментов отличаются.
# Сегмент "Средний+" содержит 'средн'; "Малый+Микро" — 'мал', 'микр'.
# Все прочие сегменты не включаются в расчёт, но отражаются в контроле.

def map_segment(value):
    s = re.sub(r"\s+", " ", str(value).casefold().replace("ё", "е")).strip()
    if s in ("", "nan", "none", "<na>"):
        return None
    if "средн" in s and "мал" not in s and "микр" not in s:
        return "Средний+"
    if "мал" in s or "микр" in s:
        return "Малый+Микро"
    return None

## 2. Загрузка и расчёт

**Важно:** проверяем именно числовые значения `NPL(договор)`: `0` — не NPL, любое другое число — NPL. На выходе дополнительно контролируются противоречивые значения NPL по разным строкам одного договора, пропуски ключей, даты и задолженность.


In [ ]:
import calendar
from datetime import datetime


def canonical_col(name):
    """Column matching robust to line breaks, dots, NBSP and inconsistent spaces."""
    return re.sub(r"[^a-zа-я0-9]+", "", str(name).casefold().replace("ё", "е"))


# Белый список. Ни один другой столбец из исходных файлов не загружается.
INPUT_COLUMNS = [
    "УНП", "Клиент", "Сегмент", "Номер договора",
    "Тип операции", "Код валюты", "ГР %", "%рез",
    "проср задолж, тыс BYN", "Задолженность, тыс BYN",
    "Резерв факт", "Резерв рассчит", "Курс",
    "Max-Day OD", "Max-Day %%", "Дата начала",
    "Дата окончания", "NPL(договор)",
]

# Семь колонок, без которых переходы и когорты посчитать нельзя.
REQUIRED = [
    "УНП", "Клиент", "Сегмент", CONTRACT_COL,
    "Задолженность, тыс BYN", "Дата начала", "NPL(договор)",
]
# Все прочие из белого списка можно не иметь в конкретном файле.
OPTIONAL = [c for c in INPUT_COLUMNS if c not in REQUIRED]

# Синонимы названий — только для разрешённых колонок.
COLUMN_ALIASES = {
    "проср задолж, тыс BYN": ["Просроченная задолженность, тыс. BYN", "Просроченная задолженность, тыс BYN"],
    "ГР %": ["ГР"],
    "%рез": ["% рез", "% рез.", "% резервирования"],
}

COL_LOOKUP = {canonical_col(c): c for c in INPUT_COLUMNS}
for standard, aliases in COLUMN_ALIASES.items():
    for alias in aliases:
        key = canonical_col(alias)
        if key in COL_LOOKUP and COL_LOOKUP[key] != standard:
            raise RuntimeError(f"Конфликт синонимов колонок: {alias}")
        COL_LOOKUP[key] = standard

assert len(INPUT_COLUMNS) == 18


def month_from_filename(path):
    if path.name in FILE_MONTH_OVERRIDE:
        return pd.Period(FILE_MONTH_OVERRIDE[path.name], freq="M")
    name = path.stem
    # yyyy-mm[-dd]; yyyy_mm[_dd]; yyyy.mm.dd
    m = re.search(r"(?<!\d)(20\d{2})[._ -](0?[1-9]|1[0-2])(?:[._ -](?:[0-2]?\d|3[01]))?(?!\d)", name)
    if m:
        return pd.Period(year=int(m.group(1)), month=int(m.group(2)), freq="M")
    # dd-mm-yyyy
    m = re.search(r"(?<!\d)(?:[0-2]?\d|3[01])[._ -](0?[1-9]|1[0-2])[._ -](20\d{2})(?!\d)", name)
    if m:
        return pd.Period(year=int(m.group(2)), month=int(m.group(1)), freq="M")
    raise ValueError(f"Не удалось определить месяц: {path.name}. Добавьте файл в FILE_MONTH_OVERRIDE")


def list_portfolios(directory):
    files = [p for p in directory.iterdir() if p.suffix.lower() in (".xlsx", ".xlsm")
             and not p.name.startswith("~$") and p.name != OUTPUT_FILE.name]
    if not files:
        raise FileNotFoundError(f"В папке нет файлов .xlsx / .xlsm: {directory}")
    result = {}
    for path in files:
        period = month_from_filename(path)
        if period in result:
            raise ValueError(f"Два файла для {period}: {result[period].name}, {path.name}. Оставьте один срез.")
        result[period] = path
    return dict(sorted(result.items()))


def clean_key(s):
    """Preserve text identifiers; remove spurious .0 only from numeric-like IDs."""
    s = s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    s = s.str.replace(r"^(\d+)\.0$", r"\1", regex=True)
    return s.mask(s.str.casefold().isin(["", "nan", "none", "<na>", "-"]))


def numeric_debt(series, file_name):
    """Parse numbers with thousand spaces and comma decimal mark."""
    text = series.astype("string").str.strip()
    text = text.str.replace(r"[\u00a0\u202f\s]", "", regex=True)
    # European '1.234,56' and standard '1234.56' are both supported.
    both = text.str.contains(",", regex=False, na=False) & text.str.contains(".", regex=False, na=False)
    text = text.mask(both, text.str.replace(".", "", regex=False))
    text = text.str.replace(",", ".", regex=False)
    values = pd.to_numeric(text, errors="coerce")
    invalid = series.notna() & text.notna() & text.ne("") & values.isna()
    if invalid.any():
        samples = series.loc[invalid].astype(str).unique()[:5].tolist()
        raise ValueError(f"Некорректная задолженность в {file_name}: {samples}; строк: {int(invalid.sum())}")
    # Missing debt means 0; count it in the quality report.
    return values.fillna(0).astype("float64"), int(values.isna().sum())


def npl_flag(series, file_name):
    """0 = not NPL, every other *numeric* value = NPL; reject blanks/text."""
    s = series.astype("string").str.strip()
    s = s.str.replace(r"[\u00a0\u202f\s]", "", regex=True)
    s = s.str.replace(",", ".", regex=False)
    numbers = pd.to_numeric(s, errors="coerce")
    invalid = numbers.isna() | ~np.isfinite(numbers.astype("float64"))
    if invalid.any():
        samples = series.loc[invalid].astype("string").fillna("<пусто>").unique()[:8].tolist()
        raise ValueError(
            f"{file_name}: в колонке NPL(договор) найдены пустые или нечисловые значения "
            f"({int(invalid.sum())} строк). Примеры: {samples}. "
            "Ожидаются только числа: 0 = нет NPL, !=0 = NPL."
        )
    return numbers.ne(0).astype(bool)


def read_portfolio(path):
    """Read only the allowed 18 columns, ignoring all other columns in the workbook."""
    raw = pd.read_excel(
        path,
        sheet_name=SHEET_NAME,
        header=HEADER_ROW,
        usecols=lambda name: canonical_col(name) in COL_LOOKUP,
    )
    raw.columns = [COL_LOOKUP[canonical_col(c)] for c in raw.columns]

    # Avoid double counting if a workbook contains two variations of the same field.
    duplicates = raw.columns[raw.columns.duplicated()].unique().tolist()
    if duplicates:
        raise ValueError(f"{path.name}: дубли колонок после нормализации: {duplicates}")

    missing = [col for col in REQUIRED if col not in raw.columns]
    if missing:
        raise KeyError(f"{path.name}: отсутствуют обязательные столбцы: {missing}. "
                       f"Найдены разрешённые колонки: {list(raw.columns)}")
    missing_optional = [col for col in OPTIONAL if col not in raw.columns]

    loaded_columns_count = len(raw.columns)
    orig_count = len(raw)
    raw["УНП"] = clean_key(raw["УНП"])
    raw[CONTRACT_COL] = clean_key(raw[CONTRACT_COL])
    missing_key = raw["УНП"].isna() | raw[CONTRACT_COL].isna()
    raw = raw.loc[~missing_key].copy()

    raw["_segment"] = raw["Сегмент"].map(map_segment)
    raw["_npl"] = npl_flag(raw["NPL(договор)"], path.name)
    raw["_date"] = pd.to_datetime(raw["Дата начала"], dayfirst=True, errors="coerce")
    raw["_balance"], missing_balance = numeric_debt(raw["Задолженность, тыс BYN"], path.name)

    keys = ["УНП", CONTRACT_COL]
    gb = raw.groupby(keys, sort=False, dropna=False)
    grouped = gb.agg(
        balance=("_balance", "sum"),
        npl=("_npl", "any"),
        orig_date=("_date", "min"),
        segment=("_segment", "first"),
        segments_count=("_segment", "nunique"),
        rows=("_balance", "size"),
        npl_states=("_npl", "nunique"),
    ).reset_index()
    grouped["Клиент"] = gb["Клиент"].first().to_numpy()
    conflicts = int((grouped["segments_count"] > 1).sum())
    if conflicts:
        example = grouped.loc[grouped["segments_count"] > 1, keys].head(5).to_dict("records")
        raise ValueError(f"{path.name}: договоров с конфликтующими сегментами: {conflicts}. Примеры: {example}")
    conflicting_npl = int((grouped["npl_states"] > 1).sum())
    grouped = grouped.drop(columns=["segments_count", "npl_states"])

    quality = {
        "Файл": path.name,
        "Разрешённых колонок загружено (из 18)": loaded_columns_count,
        "Необязательные колонки отсутствуют": ", ".join(missing_optional) or "—",
        "Строк исходного портфеля": orig_count,
        "Строк без ключа (исключены)": int(missing_key.sum()),
        "Договоров после агрегации": len(grouped),
        "Строк с пустой задолженностью (=0)": missing_balance,
        "Договоров без сегмента из 2 групп": int(grouped["segment"].isna().sum()),
        "Договоров без даты начала": int(grouped["orig_date"].isna().sum()),
        "Договоров в NPL": int(grouped["npl"].sum()),
        "Договоров с NPL=0 и NPL!=0 в разных строках": conflicting_npl,
    }
    raw_segments = raw["Сегмент"].astype("string").value_counts(dropna=False).head(30)
    raw_npl = raw["NPL(договор)"].astype("string").value_counts(dropna=False).head(30)
    return grouped, quality, raw_segments, raw_npl


def process_month(previous, current, month):
    keys = ["УНП", CONTRACT_COL]
    old = previous[keys + ["npl", "balance", "orig_date"]].rename(columns={
        "npl": "npl_previous", "balance": "balance_previous", "orig_date": "orig_date_previous"
    })
    new = current[keys + ["npl", "balance", "orig_date", "segment", "Клиент"]].rename(columns={
        "npl": "npl_current", "balance": "balance_current", "orig_date": "orig_date_current"
    })
    merged = new.merge(old, on=keys, how="left", indicator=True, validate="one_to_one")
    exists_before = merged["_merge"].eq("both")
    new_npl_without_previous = int((merged["npl_current"] & ~exists_before).sum())
    transitioned = exists_before & ~merged["npl_previous"].eq(True) & merged["npl_current"]
    events = merged.loc[transitioned].copy()

    events["orig_date"] = events["orig_date_current"].combine_first(events["orig_date_previous"])
    events["cohort_year"] = events["orig_date"].dt.year.astype("Int64")
    invalid_year = events["cohort_year"].isna() | ~events["cohort_year"].between(MIN_COHORT_YEAR, MAX_COHORT_YEAR)
    future_date = events["orig_date"].gt(month.to_timestamp(how="end"))
    wrong_segment = events["segment"].isna()
    eligible = ~invalid_year & ~future_date & ~wrong_segment & (events["cohort_year"] <= month.year)
    selected = events.loc[eligible].copy()
    selected["month"] = str(month)
    selected["event_year"] = month.year

    controls = {
        "Месяц": str(month),
        "Договоров на начало": len(previous),
        "Договоров на конец": len(current),
        "NPL на конец без договора на начало (не учитываем)": new_npl_without_previous,
        "Переходов 0→1 всего": len(events),
        "Из них без года 2023–2026": int(invalid_year.sum()),
        "Из них вне нужных сегментов": int((~invalid_year & wrong_segment).sum()),
        "Из них с датой позже конца месяца": int((~invalid_year & future_date).sum()),
        "Переходов включено в расчёт": len(selected),
        "Сумма включённых, тыс. BYN": round(float(selected["balance_current"].sum()), 2),
    }
    return selected, controls

## 3. Запуск и выгрузка

Выходной Excel содержит листы: **Свод_месяцы** (все сочетания месяц × год заключения × сегмент), **Матрица_когорт**, **Годовые_итоги**, **Договоры_переходы** (каждое событие на уровне договора), **Контроль_месяцы**, **Контроль_файлы**. Если нет нужного месячного портфеля, нуля вместо отсутствующего сравнения не будет.


In [ ]:
from collections import Counter

file_map = list_portfolios(DATA_DIR)
print(f"Найдено портфелей: {len(file_map)}. Период: {min(file_map)} — {max(file_map)}")

all_events = []
month_controls = []
file_controls = []
analyzed_months = []
prev = None
prev_month = None

for month, file_path in file_map.items():
    current, quality, segments, npl_values = read_portfolio(file_path)
    quality["Месяц"] = str(month)
    file_controls.append(quality)
    if prev is not None and month == (prev_month + 1) and MIN_COHORT_YEAR <= month.year <= MAX_COHORT_YEAR:
        ev, ctrl = process_month(prev, current, month)
        month_controls.append(ctrl)
        analyzed_months.append(month)
        if len(ev):
            all_events.append(ev)
        print(f"{month}: выходов в NPL = {ctrl['Переходов включено в расчёт']:,}, "
              f"сумма = {ctrl['Сумма включённых, тыс. BYN']:,.2f} тыс. BYN")
    elif MIN_COHORT_YEAR <= month.year <= MAX_COHORT_YEAR:
        print(f"{month}: ПРОПУЩЕН: отсутствует предыдущий месяц ({month - 1})")
    prev, prev_month = current, month
    if month == min(file_map):
        print("Сегменты (первый файл):", segments.head(20).to_dict())
        print("Значения NPL (первый файл):", npl_values.head(20).to_dict())

if not analyzed_months:
    raise RuntimeError("Нет ни одной пары соседних месячных портфелей. Проверьте имена файлов.")

if all_events:
    event_df = pd.concat(all_events, ignore_index=True)
else:
    event_df = pd.DataFrame(columns=["month", "event_year", "cohort_year", "segment", "УНП", CONTRACT_COL,
                                     "Клиент", "orig_date", "balance_current", "balance_previous"])

# Красивая детализация — договор появляется один раз для каждого перехода 0→1.
detail = event_df.rename(columns={
    "month": "Месяц выхода",
    "event_year": "Год выхода",
    "cohort_year": "Год заключения договора",
    "segment": "Сегмент",
    "orig_date": "Дата начала договора",
    "balance_previous": "Задолженность на начало, тыс BYN",
    "balance_current": "Задолженность на конец, тыс BYN",
})
detail["NPL на начало"] = False
detail["NPL на конец"] = True
detail_cols = ["Месяц выхода", "Год выхода", "Год заключения договора", "Сегмент",
               "УНП", CONTRACT_COL, "Клиент", "Дата начала договора",
               "Задолженность на начало, тыс BYN", "Задолженность на конец, тыс BYN",
               "NPL на начало", "NPL на конец"]
detail = detail.reindex(columns=detail_cols)
if len(detail):
    detail = detail.sort_values(["Месяц выхода", "Сегмент", "Год заключения договора", "УНП", CONTRACT_COL])

# Сетка существующих пар месячных срезов, чтобы 0 отличался от «нет данных».
segments_order = ["Средний+", "Малый+Микро"]
all_combinations = [
    (str(m), m.year, year, segment)
    for m in analyzed_months
    for year in range(MIN_COHORT_YEAR, min(MAX_COHORT_YEAR, m.year) + 1)
    for segment in segments_order
]
full = pd.DataFrame(all_combinations, columns=["month", "event_year", "cohort_year", "segment"])
if len(event_df):
    sums = (event_df.groupby(["month", "event_year", "cohort_year", "segment"], observed=True)
            .agg(sum_debt=("balance_current", "sum"), contracts=(CONTRACT_COL, "size"))
            .reset_index())
    sums["cohort_year"] = sums["cohort_year"].astype("int64")
    full = full.merge(sums, on=["month", "event_year", "cohort_year", "segment"], how="left", validate="one_to_one")
else:
    full["sum_debt"] = np.nan
    full["contracts"] = np.nan
full["sum_debt"] = full["sum_debt"].fillna(0).round(2)
full["contracts"] = full["contracts"].fillna(0).astype("int64")

summary = full.rename(columns={
    "month": "Месяц выхода", "event_year": "Год выхода",
    "cohort_year": "Год заключения договора", "segment": "Сегмент",
    "sum_debt": "Сумма выхода в NPL, тыс BYN", "contracts": "Количество договоров"
})
summary = summary[["Месяц выхода", "Сегмент", "Год заключения договора", "Год выхода",
                   "Сумма выхода в NPL, тыс BYN", "Количество договоров"]]

matrix = summary.pivot(index=["Сегмент", "Год заключения договора"], columns="Месяц выхода",
                       values="Сумма выхода в NPL, тыс BYN")
full_index = pd.MultiIndex.from_product([segments_order, range(MIN_COHORT_YEAR, MAX_COHORT_YEAR + 1)],
                                         names=["Сегмент", "Год заключения договора"])
matrix = matrix.reindex(index=full_index, columns=[str(x) for x in analyzed_months]).reset_index()

annual = (summary.groupby(["Сегмент", "Год заключения договора", "Год выхода"], as_index=False)
          .agg(**{"Сумма выхода в NPL, тыс BYN": ("Сумма выхода в NPL, тыс BYN", "sum"),
                 "Количество договоров": ("Количество договоров", "sum")}))
annual["Сумма выхода в NPL, тыс BYN"] = annual["Сумма выхода в NPL, тыс BYN"].round(2)

control_months = pd.DataFrame(month_controls)
control_files = pd.DataFrame(file_controls)

# Проверка суммы из строк договоров против суммы сводной.
assert np.isclose(detail["Задолженность на конец, тыс BYN"].sum(),
                  summary["Сумма выхода в NPL, тыс BYN"].sum(), atol=0.02)
assert detail.shape[0] == summary["Количество договоров"].sum()

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for sheet_name, frame in [
        ("Свод_месяцы", summary), ("Матрица_когорт", matrix),
        ("Годовые_итоги", annual), ("Договоры_переходы", detail),
        ("Контроль_месяцы", control_months), ("Контроль_файлы", control_files),
    ]:
        frame.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "C2" if sheet_name == "Матрица_когорт" else "A2"
        ws.auto_filter.ref = ws.dimensions
        from openpyxl.styles import Font, PatternFill, Alignment
        from openpyxl.utils import get_column_letter
        for cell in ws[1]:
            cell.fill = PatternFill(fill_type="solid", fgColor="183153")
            cell.font = Font(color="FFFFFF", bold=True)
            cell.alignment = Alignment(wrap_text=True, vertical="center")
        ws.row_dimensions[1].height = 33
        for idx, col in enumerate(frame.columns, start=1):
            width = max(13, min(39, len(str(col)) + 2))
            if col in ("УНП", CONTRACT_COL):
                width = 24
            if "Месяц" in str(col) or re.fullmatch(r"20\d{2}-\d{2}", str(col)):
                width = 14
            ws.column_dimensions[get_column_letter(idx)].width = width
            if "тыс BYN" in str(col) or "тыс. BYN" in str(col) or re.fullmatch(r"20\d{2}-\d{2}", str(col)):
                for cells in ws.iter_cols(min_col=idx, max_col=idx, min_row=2):
                    for cell in cells:
                        cell.number_format = '#,##0.00;[Red](#,##0.00);–'
            if "Дата начала" in str(col):
                for cells in ws.iter_cols(min_col=idx, max_col=idx, min_row=2):
                    for cell in cells:
                        cell.number_format = "DD.MM.YYYY"

print(f"\nГотово: {OUTPUT_FILE}")
print(f"Проверено месяцев: {len(analyzed_months)}; событий: {len(detail):,}; "
      f"сумма: {detail['Задолженность на конец, тыс BYN'].sum():,.2f} тыс. BYN")
display(summary.head(16))

### Как читать результат

- **Год заключения договора** = год `Дата начала` (при нескольких строках договора берётся самая ранняя непустая дата).
- **Месяц выхода** = месяц, в котором договор сменил NPL с `0` на **любое ненулевое число** относительно предыдущего месячного среза.
- **Сумма выхода в NPL, тыс BYN** = сумма поля `Задолженность, тыс BYN` **на конец месяца** по всем строкам договора.
- **Количество договоров** = число событий перехода в данном месяце / когорте / сегменте.
- **Матрица_когорт**: строка — когорта × сегмент; столбец — месяц выхода; пустые ячейки до появления когорты или при отсутствии сопоставимой пары портфелей **не равны нулю**.
- **Контроль_файлы** содержит число договоров с противоречащими друг другу NPL-флагами в разных строках одного и того же договора. Для статуса договора используется `any` по строкам, то есть наличие ненулевого NPL хотя бы в одной строке; такие договоры следует отдельно проверить.
- Столбцы `Тип операции`, `Код валюты` могут присутствовать, но **не входят в ключ договора**; поэтому все их строки участвуют в сумме задолженности по одному договору.

**Допущения:** ключ `УНП + Номер договора` однозначно идентифицирует договор. Если `Дата начала` относится к траншу, а не договору, когорта может определяться неверно. УО и внебалансовые позиции специально **не исключаются**: суммируется ровно входной столбец задолженности.
